# 🧠 Phase 1 — EEG Data Preprocessing

This notebook loads raw EEG signals from the I-CARE dataset, performs filtering, referencing, and segmentation, and exports processed tensors for downstream modeling.

### 🎯 Goals
- Load `.mat` / `.hea` EEG data  
- Apply bandpass filtering (0.5–50 Hz)  
- Remove artifacts (NaN, spikes)  
- Re-reference (average or linked ears)  
- Segment data into fixed windows (e.g., 5 s)  
- Save processed signals as `.npy` or `.pt` arrays


In [ ]:
# Parameters (Papermill injects values here)
data_path = '../data/icare/training'
output_path = '../data/icare/preprocessed'
fs = 256          # Sampling frequency
lowcut = 0.5
highcut = 45
segment_len = 5   # Segment length in seconds


: 

In [20]:
import os
import numpy as np
import mne
from scipy.io import loadmat
from pathlib import Path

Path(output_path).mkdir(parents=True, exist_ok=True)

for patient_dir in Path(data_path).iterdir():
    for file in patient_dir.glob('*_EEG.mat'):
        data = loadmat(file)
        eeg = data.get('val') if 'val' in data else np.array(list(data.values())[-1])
        eeg = np.nan_to_num(eeg)
        
        info = mne.create_info(
            ch_names=[f'Ch{i}' for i in range(eeg.shape[0])],
            sfreq=fs,
            ch_types='eeg'
        )
        raw = mne.io.RawArray(eeg, info)
        raw.filter(lowcut, highcut, fir_design='firwin')
        
        data_array = raw.get_data()
        n_samples = data_array.shape[1]
        samples_per_segment = fs * segment_len

        # ✅ Step 1: Trim data to make it divisible by segment size
        usable_len = (n_samples // samples_per_segment) * samples_per_segment
        if usable_len == 0:
            print(f"Skipping {file.name} — recording too short for {segment_len}s segments.")
            continue

        data_array = data_array[:, :usable_len]

        # ✅ Step 2: Split into equal-sized chunks
        segments = np.split(data_array, usable_len // samples_per_segment, axis=1)

        # ✅ Step 3: Stack and save
        segments = np.stack(segments)  # Shape: (num_segments, channels, samples)
        np.save(Path(output_path) / f'{file.stem}_segments.npy', segments)

        print(f"✅ Processed {file.name} — {len(segments)} segments saved.")


Creating RawArray with float64 data, n_channels=19, n_times=1578500
    Range : 0 ... 1578499 =      0.000 ...  6166.012 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 1691 samples (6.605 s)

✅ Processed 0284_001_004_EEG.mat — 1233 segments saved.
Creating RawArray with float64 data, n_channels=19, n_times=1800000
    Range : 0 ... 1799999 =      0.000 ...  7031.246 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 45 Hz

FIR filter parameters
---

✅ **Output:** Cleaned, bandpassed EEG segments saved under  
`data/icare/preprocessed/` — ready for baseline and foundation model training.


In [23]:
import numpy as np
from pathlib import Path

data_array = raw.get_data()
n_samples = data_array.shape[1]
samples_per_segment = fs * segment_len

# Pad last incomplete segment if needed
remainder = n_samples % samples_per_segment
if remainder > 0:
    pad_width = samples_per_segment - remainder
    data_array = np.pad(data_array, ((0, 0), (0, pad_width)), mode='constant')

# Split into segments and save
segments = np.split(data_array, data_array.shape[1] // samples_per_segment, axis=1)
segments = np.stack(segments)
np.save(Path(output_path) / f'{file.stem}_segments.npy', segments)

print(f"✅ Processed {file.name} → {len(segments)} segments (padded).")



✅ Processed 0424_001_003_EEG.mat → 176 segments (padded).
